# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullahhashmi01/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

queue = pd.read_csv(file_name)

print("Queue loaded successfully.")
print("Rows:", len(queue))
print("Columns:", queue.columns.tolist())

Saving content_refresh_anonymized.csv to content_refresh_anonymized (1).csv
Queue loaded successfully.
Rows: 30000
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [12]:
import numpy as np
import pandas as pd

# ── 1. reason_code banao existing columns se ──────────────────────
def build_reason_code(row):
    codes = []

    # High impressions check
    if str(row.get("impression_tier", "")).lower() in ["good", "high", "excellent"]:
        codes.append("HIGH_IMPRESSIONS")

    # Low CTR check
    if pd.notna(row.get("ctr")) and row.get("ctr", 99) < 1.0:
        codes.append("LOW_CTR")

    # Position tier check
    pos = str(row.get("position_tier", "")).lower()
    if "striking" in pos:
        codes.append("POSITION_4_TO_20")
    elif "page_3" in pos or "page_4" in pos or "page_5" in pos:
        codes.append("POSITION_21_TO_40")

    # Missing position check
    avg_pos = row.get("avg_position", 0)
    if pd.isna(avg_pos) or avg_pos == 0:
        codes.append("MISSING_POSITION")

    return "|".join(codes) if codes else "NO_SIGNAL"

queue["reason_code"] = queue.apply(build_reason_code, axis=1)


# ── 2. action_score banao (0 to 100) ──────────────────────────────
# Zyada score = zyada urgent action chahiye

# Trend score: zyada decline = zyada urgent
trend_score = queue["trend_pct"].clip(upper=0).abs()  # 0 to inf
trend_score = (trend_score / trend_score.max()) * 50  # 0 to 50

# Impression score: zyada impressions = zyada impact
imp_score = queue["impressions_90d"].fillna(0)
imp_score = (imp_score / imp_score.max()) * 30        # 0 to 30

# CTR score: kam CTR = zyada opportunity
ctr_score = (1 - (queue["ctr"].clip(0, 10) / 10)) * 20  # 0 to 20

queue["action_score"] = (
    trend_score + imp_score + ctr_score
).round(2)


# ── 3. Action choose karne ka function ────────────────────────────
def choose_action(reason):
    reason = str(reason).upper()

    if "HIGH_IMPRESSIONS" in reason and "LOW_CTR" in reason:
        return "Review title, description and search-intent alignment"

    if "POSITION_4_TO_20" in reason:
        return "Review content relevance, depth and internal links"

    if "POSITION_21_TO_40" in reason:
        return "Investigate relevance before planning a larger refresh"

    if "MISSING_POSITION" in reason:
        return "Verify search data before taking editorial action"

    return "Keep in the review backlog and collect more evidence"

queue["recommended_action"] = queue["reason_code"].apply(choose_action)


# ── 4. Sort aur rank ──────────────────────────────────────────────
queue = (
    queue
    .sort_values("action_score", ascending=False)
    .reset_index(drop=True)
)
queue["rank"] = np.arange(1, len(queue) + 1)


# ── 5. Top 20 display karo ────────────────────────────────────────
display(
    queue[[
        "rank",
        "content_id",
        "action_score",
        "reason_code",
        "recommended_action"
    ]].head(20)
)

,rank,content_id,action_score,reason_code,recommended_action
0,1,content_66b4046cc144,77.19,HIGH_IMPRESSIONS|LOW_CTR|POSITION_21_TO_40,"Review title, description and search-intent al..."
1,2,content_5fe46e04994d,72.12,HIGH_IMPRESSIONS|LOW_CTR,"Review title, description and search-intent al..."
2,3,content_8c19996aa890,71.46,HIGH_IMPRESSIONS|LOW_CTR,"Review title, description and search-intent al..."
3,4,content_33da44cb09c9,70.73,HIGH_IMPRESSIONS|LOW_CTR,"Review title, description and search-intent al..."
4,5,content_20e876d26019,70.43,HIGH_IMPRESSIONS|LOW_CTR|POSITION_21_TO_40,"Review title, description and search-intent al..."
5,6,content_ec66c58d9826,70.21,HIGH_IMPRESSIONS|LOW_CTR,"Review title, description and search-intent al..."
6,7,content_581f405133c1,70.16,HIGH_IMPRESSIONS|LOW_CTR|POSITION_4_TO_20,"Review title, description and search-intent al..."
7,8,content_e74933316051,70.10,LOW_CTR,Keep in the review backlog and collect more ev...
8,9,content_331a01416e92,70.09,HIGH_IMPRESSIONS|LOW_CTR,"Review title, description and search-intent al..."
9,10,content_6eb16d7a88ae,70.07,LOW_CTR,Keep in the review backlog and collect more ev...


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
human_review = pd.DataFrame({
    "review_check": [
        "Search intent verified",
        "Business importance checked",
        "Recent edits checked",
        "Seasonality checked",
        "Measurements complete",
        "Cannibalisation checked",
        "Legal or brand risk checked",
        "Final human decision recorded"
    ],
    "required_before_action": [
        True,
        True,
        True,
        True,
        True,
        True,
        True,
        True
    ]
})

no_go_actions = pd.DataFrame({
    "never_automate": [
        "Publish or rewrite content",
        "Delete or redirect a page",
        "Change sensitive factual claims",
        "Act with missing critical data",
        "Expose or infer private identities",
        "Promise ranking or traffic improvement"
    ],
    "reason": [
        "Requires editorial approval",
        "May cause irreversible traffic loss",
        "Requires qualified human review",
        "Recommendation may be unreliable",
        "Violates privacy requirements",
        "The evidence is observational"
    ]
})

print("Required human-review checks:")
display(human_review)

print("No-go actions:")
display(no_go_actions)

Required human-review checks:


,review_check,required_before_action
0,Search intent verified,True
1,Business importance checked,True
2,Recent edits checked,True
3,Seasonality checked,True
4,Measurements complete,True
5,Cannibalisation checked,True
6,Legal or brand risk checked,True
7,Final human decision recorded,True


No-go actions:


,never_automate,reason
0,Publish or rewrite content,Requires editorial approval
1,Delete or redirect a page,May cause irreversible traffic loss
2,Change sensitive factual claims,Requires qualified human review
3,Act with missing critical data,Recommendation may be unreliable
4,Expose or infer private identities,Violates privacy requirements
5,Promise ranking or traffic improvement,The evidence is observational


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
monitoring_triggers = pd.DataFrame({
    "measurement": [
        "Precision@20",
        "Missing-position rate",
        "Median feature value",
        "Required column",
        "New population share",
        "Queue age"
    ],
    "trigger": [
        "More than 20% below validation result",
        "Increase of at least 10 percentage points",
        "Change of more than 25%",
        "Missing or changed data type",
        "More than 10% from a new group",
        "More than 30 days old"
    ],
    "response": [
        "Review errors and retrain if needed",
        "Check data collection",
        "Investigate feature drift",
        "Stop scoring and repair pipeline",
        "Run a new grouped validation",
        "Rebuild the ranked queue"
    ]
})

display(monitoring_triggers)

,measurement,trigger,response
0,Precision@20,More than 20% below validation result,Review errors and retrain if needed
1,Missing-position rate,Increase of at least 10 percentage points,Check data collection
2,Median feature value,Change of more than 25%,Investigate feature drift
3,Required column,Missing or changed data type,Stop scoring and repair pipeline
4,New population share,More than 10% from a new group,Run a new grouped validation
5,Queue age,More than 30 days old,Rebuild the ranked queue


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

def confidence_note(reason):

    reason = str(reason)

    if "MISSING_POSITION" in reason:
        return "Low: position data is missing"

    strong_codes = [
        "HIGH_IMPRESSIONS",
        "LOW_CTR",
        "POSITION_4_TO_20"
    ]

    signal_count = sum(
        code in reason
        for code in strong_codes
    )

    if signal_count >= 3:
        return "Higher: three rule signals agree"

    if signal_count == 2:
        return "Medium: two rule signals agree"

    return "Low: limited rule evidence"


def wrong_if_note(reason):

    reason = str(reason)

    if "MISSING_POSITION" in reason:
        return (
            "Search-position measurements "
            "are incomplete"
        )

    if "LOW_CTR" in reason:
        return (
            "Low CTR is normal for the "
            "observed search intent"
        )

    if "POSITION_4_TO_20" in reason:
        return (
            "The position is temporary or "
            "demand is seasonal"
        )

    return (
        "Business context does not support "
        "editorial action"
    )


queue["confidence_note"] = (
    queue["reason_code"].apply(
        confidence_note
    )
)

queue["what_could_make_it_wrong"] = (
    queue["reason_code"].apply(
        wrong_if_note
    )
)

In [17]:
print("PLAYBOOK CHECK")

print("Ranked rows:", len(paper_queue))
print("Top-20 rows:", len(top_20_review))

print(
    "Missing actions:",
    paper_queue[
        "recommended_action"
    ].isna().sum()
)

print(
    "Private client-name column included:",
    "client_name" in paper_queue.columns
)

print(
    "URL column included:",
    "url" in paper_queue.columns
)

print(
    "Query column included:",
    "query" in paper_queue.columns
)

print("\nML-10 exports completed.")

PLAYBOOK CHECK
Ranked rows: 30000
Top-20 rows: 20
Missing actions: 0
Private client-name column included: False
URL column included: False
Query column included: False

ML-10 exports completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.